> ## SOLUTION / ANSWER KEY &mdash; Lab 6.3
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-03-model-and-prompt.ipynb`](../lab-03-model-and-prompt.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.3 &mdash; The Model Interface: Prompts & .invoke()

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Build a PromptTemplate with an {input} slot and format it
- Configure a ChatOllama model (the .invoke().content interface)
- See that the SAME interface works for any model (model-agnostic)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; LangChain's core building blocks](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-03"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Every LangChain chat model shares **one interface**: `model.invoke(prompt)` returns a message whose
text is in **`.content`**. That is why swapping **`ChatOllama`** for **`ChatGroq`** is a one-line
change. A **`PromptTemplate`** (from `langchain_core.prompts`) fills slots like `{input}`. Building the
prompt and configuring the model are **deterministic** &mdash; the actual call to the LLM is the one
step that needs a running model, so we do it in the optional cell.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

demo_prompt = PromptTemplate.from_template("Say hi to {who}.")
print("formatted:", demo_prompt.format(who="Ada"))
demo_model = ChatOllama(model="llama3.2:1b")
print("model configured:", demo_model.model, "| has .invoke:", callable(getattr(demo_model, "invoke", None)))

## Your Turn
Build the **template** (with an `{input}` slot), format it in `build_prompt`, and configure the chat model. (The live call is the optional cell.)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

def build_template():
    return PromptTemplate.from_template("Answer concisely.\nQuestion: {input}")

def build_prompt(question):
    return build_template().format(input=question)

def build_model():
    return ChatOllama(model="llama3.2:1b")

try:
    print('input variables:', build_template().input_variables)
    print(build_prompt('what is an agent?'))
    m = build_model()
    print('---')
    print('model:', m.model, '| same .invoke().content interface as ChatGroq')
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the template exposes exactly the {input} variable", lambda: build_template().input_variables == ["input"])
expect_true("the prompt fills in the question", lambda: "capital of france" in build_prompt("capital of france"))
expect_true("a DIFFERENT question is substituted, not left as a slot", lambda: "photosynthesis" in build_prompt("photosynthesis") and "{input}" not in build_prompt("photosynthesis"))
expect_true("the prompt keeps its Question: label", lambda: "Question:" in build_prompt("x"))
expect_true("the model is a ChatOllama for llama3.2:1b", lambda: build_model().model == "llama3.2:1b")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Format a prompt, then actually call the model and read `.content`. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        model = build_model()
        reply = model.invoke(build_prompt("In one sentence, what is an AI agent?"))
        print("reply:", reply.content)
    else:
        print("No Ollama reachable -- skipping the live call. The prompt/model above are already built.")
except Exception as e:
    print("Live call skipped:", type(e).__name__, "(install langchain-ollama + run `ollama run llama3.2:1b`).")

---
### Key takeaway
One interface -- .invoke(prompt).content -- over every provider. Building the prompt and configuring the model is deterministic; only the call itself needs a running LLM.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>